# Advanced Logistic Regression: Softmax Regression, Non-Linear Classification & Scikit-Learn Hyperparameters

Standard binary Logistic Regression is a powerful classification algorithm, but real-world scenarios often present more complex challenges, such as multiple output categories, non-linear relationships, and custom regularization requirements.

In these notes, we will explore:
1. **Softmax Regression (Multinomial Logistic Regression):** Extending standard logistic regression to multi-class classification.
2. **Polynomial Features for Non-Linear Classification:** Transforming feature space to capture curved decision boundaries.
3. **Scikit-Learn Logistic Regression Hyperparameters Guide:** An in-depth reference for regularization, solvers, and class weighting settings.
4. **Hands-on Python Demos:** Step-by-step implementation of multi-class classification, polynomial boundaries, and hyperparameter grid tuning.

## 1. Softmax Regression (Multinomial Logistic Regression)

### The Problem
Standard Logistic Regression is strictly a **binary classifier**. In real-world tasks, we often need to predict one out of $K > 2$ classes (e.g. classifying flower types, predicting student career paths, or recognizing handwritten digits). A generalized mathematical approach is required.

### The Softmax Function
The **Softmax function** is the mathematical engine of multi-class logistic regression. Given a vector of raw prediction scores (called **logits**) $z = [z_1, z_2, \dots, z_K]^T$ computed for each class, Softmax converts them into a probability distribution:

$$P(y = c | X) = \hat{y}_c = \frac{e^{z_c}}{\sum_{j=1}^K e^{z_j}}$$

#### Crucial Properties:
- **Normalization:** The sum of all class probabilities always equals exactly $1$:
  $$\sum_{c=1}^K P(y = c | X) = 1$$
- **Range Constraint:** All output probabilities are strictly bounded:
  $$0 \le P(y = c | X) \le 1$$
- **Exponential Scaling:** The exponential terms ($e^{z_c}$) amplify differences, causing the largest input logit to dominate the probability distribution.

### Training Intuition
How do we optimize weights for multiple classes?
- **The Naive Approach:** You could one-hot encode the target column into $K$ binary columns, train $K$ separate binary logistic regression models (One-vs-Rest), and aggregate their scores. This is computationally expensive.
- **The Unified Approach:** Softmax regression trains a single unified model. It constructs a weight vector $W_c$ for **each class $c$**. For $K$ classes, the parameter matrix $W$ contains weights for all classes. We optimize all weights simultaneously using a single loss function.

### Prediction
For a new input vector $X$:
1. Calculate raw logits for each class: $z_c = W_c^T X$ (where $W_c$ includes the bias term).
2. Pass logits through the Softmax function to retrieve probabilities: $\hat{y}_c = \text{Softmax}(z_c)$.
3. Predict the final label using the **argmax** function (the class with the highest probability):
   $$\hat{y} = \arg\max_{c} P(y = c | X)$$

### The Loss Function: Categorical Cross-Entropy
To train the unified multi-class model, we generalize binary cross-entropy to **Categorical Cross-Entropy (CCE)**:

$$L(W) = -\frac{1}{n} \sum_{i=1}^{n} \sum_{c=1}^{K} y_{ic} \log(\hat{y}_{ic})$$

Where:
- $n$ is the total number of samples.
- $K$ is the number of classes.
- $y_{ic}$ is the actual target indicator ($1$ if sample $i$ belongs to class $c$, $0$ otherwise - i.e., the target is one-hot encoded).
- $\hat{y}_{ic}$ is the model's predicted probability that sample $i$ belongs to class $c$.

During training, Gradient Descent calculates the partial derivatives of this loss w.r.t. all parameters in $W$ to iteratively adjust the classification boundaries.

## 2. Non-Linear Classification via Polynomial Features

### The Linear Decision Boundary Limitation
By default, standard Logistic Regression computes a linear combination of input features:
$$z = w_0 + w_1 x_1 + w_2 x_2 = 0$$
This forces the decision boundary to be a **straight line** (in 2D) or a **hyperplane** (in higher dimensions). 
- **The Problem:** If data clusters are concentric circles, nested curves, or spiral shapes, a straight line will fail to partition the classes, leading to severe **underfitting**.

### The Solution: Feature Expansion
To fit complex patterns, we can transform our input features into a higher-degree polynomial space before training.
If we have two original features $X = [x_1, x_2]$ and we apply a **polynomial expansion of degree 2**, we generate:
$$\Phi(X) = [1, x_1, x_2, x_1^2, x_2^2, x_1 x_2]$$

Logistic Regression then trains on these expanded features:
$$z = w_0 + w_1 x_1 + w_2 x_2 + w_3 x_1^2 + w_4 x_2^2 + w_5 x_1 x_2 = 0$$

Although the model is still mathematically linear with respect to the *weights* $W$, the resulting decision boundary in the original 2D space ($x_1, x_2$) becomes a **curved shape** (an ellipse, circle, parabola, or hyperbola), allowing perfect separation.

### Balancing Complexity
- **Low Degree (e.g., Degree 1):** Too simple, causes **underfitting** on curved data.
- **High Degree (e.g., Degree 7+):** Too complex, causes the model to chase outlier noise, leading to **overfitting** (poor generalization).
- **Alternatives:** While polynomial features work, manual feature engineering is often tedious. For highly complex non-linear classification, algorithms like **Support Vector Machines (SVMs)** with kernel tricks, **Decision Trees**, **Random Forests**, or **Neural Networks** are preferred.

## 3. Scikit-Learn Logistic Regression Hyperparameters Guide

Scikit-Learn's `LogisticRegression` class offers a wide array of hyperparameters to customize optimization and prevent overfitting.

### Key Hyperparameters

1. **`penalty`:** Regularization type.
   - `'l2'` (Default): Adds Ridge penalty ($\sum W^2$). Shrinks weights, preventing extreme values.
   - `'l1'`: Adds Lasso penalty ($\sum |W|$). Enforces sparsity (shrinks some weights to exactly 0), performing feature selection.
   - `'elasticnet'`: Combines L1 and L2 penalties.
   - `None`: No regularization (standard maximum likelihood).
2. **`C`:** Inverse of regularization strength (Default: `1.0`).
   - Small $C$ (e.g., $0.01$): Strong regularization, simpler model, prevents overfitting.
   - Large $C$ (e.g., $100.0$): Weak regularization, complex model, fits training data closely.
3. **`solver`:** The mathematical algorithm used to optimize weights.
   - `'lbfgs'` (Default): Limited-memory BFGS. Excellent for small/medium datasets. Supports L2 or no penalty.
   - `'liblinear'`: Good for small datasets. Supports L1 and L2, but does not support multinomial loss directly.
   - `'newton-cg'`: Newton conjugate gradient. Supports L2 or no penalty. Good for multiclass.
   - `'sag'` / `'saga'`: Stochastic Average Gradient Descent. Fast for large datasets. `'saga'` is highly versatile as it supports L1, L2, ElasticNet, and no penalty.
4. **`max_iter`:** Maximum iterations for the solver to converge (Default: `100`). Increase if you get convergence warnings.
5. **`multi_class`:** Strategy for multi-class classification.
   - `'multinomial'`: Minimizes the categorical cross-entropy loss (Softmax Regression).
   - `'ovr'`: One-vs-Rest. Fits binary classifiers for each class.
   - `'auto'` (Default): Selects `'multinomial'` if binary or if solver supports it, else `'ovr'`.
6. **`class_weight`:** Handles imbalanced datasets. Setting to `'balanced'` automatically adjusts class weights inversely proportional to class frequencies.
7. **`l1_ratio`:** Regularization mix parameter, only used when `penalty='elasticnet'`. Maps $0.0$ (strictly L2) to $1.0$ (strictly L1).

### Solver-Penalty Compatibility Matrix

Not all solvers support all penalty types. Refer to the table below when configuring models:

| Solver | L2 Penalty | L1 Penalty | ElasticNet | No Penalty (`None`) | Multi-class Support |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **`lbfgs`** | Yes | No | No | Yes | Multinomial / OVR |
| **`liblinear`** | Yes | Yes | No | No | OVR Only |
| **`newton-cg`** | Yes | No | No | Yes | Multinomial / OVR |
| **`sag`** | Yes | No | No | Yes | Multinomial / OVR |
| **`saga`** | Yes | Yes | Yes | Yes | Multinomial / OVR |

## 4. Code Setup & Imports

Let's import the necessary libraries to demonstrate these concepts.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, make_moons
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

# Set visualization contexts
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)

# Helper function to plot decision boundaries
def plot_decision_boundary(model, X, y, title, ax=None):
    if ax is None:
        ax = plt.gca()
        
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                         np.arange(y_min, y_max, 0.02))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='rainbow')
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, edgecolor='k', s=50, cmap='rainbow')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    return scatter

print("Setup completed successfully!")

## 5. Demo A: Softmax Regression on the Iris Dataset

We will fit a multi-class Logistic Regression model on the classic **Iris Dataset** (3 flower classes, using only 2 features for 2D visualization). We will configure the model to use the multinomial Softmax formulation.

In [ ]:
# 1. Load Iris Dataset
iris = load_iris()
X_iris = iris.data[:, [2, 3]]  # select petal length and petal width
y_iris = iris.target

# 2. Scale features
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

# 3. Fit Softmax Regression
softmax_reg = LogisticRegression(
    multi_class='multinomial',
    solver='lbfgs',
    penalty='l2',
    C=1.0,
    random_state=42
)
softmax_reg.fit(X_iris_scaled, y_iris)

# 4. View probability distribution for the first 3 samples
probs = softmax_reg.predict_proba(X_iris_scaled[:3])
print("Probability distributions for first 3 samples:")
for i, p in enumerate(probs):
    print(f"Sample {i+1}: Class 0: {p[0]:.4f} | Class 1: {p[1]:.4f} | Class 2: {p[2]:.4f}")

# 5. Plot decision regions
plt.figure(figsize=(10, 6))
plot_decision_boundary(softmax_reg, X_iris_scaled, y_iris, 'Softmax Regression Decision Boundaries (Iris Dataset)')
plt.xlabel('Petal Length (scaled)')
plt.ylabel('Petal Width (scaled)')
plt.tight_layout()
plt.show()

## 6. Demo B: Non-Linear Boundaries via Polynomial Features

We will generate a non-linear **make_moons** dataset and compare standard linear logistic regression against models utilizing degree 2 and degree 7 polynomial feature expansions.

In [ ]:
# 1. Generate Moon Dataset
X_moon, y_moon = make_moons(n_samples=200, noise=0.20, random_state=42)

# 2. Split and scale features
X_train, X_test, y_train, y_test = train_test_split(X_moon, y_moon, test_size=0.3, random_state=42)

# We use Scikit-Learn Pipelines to combine polynomial mapping, scaling, and training
# Model 1: Standard Linear model (Degree 1)
model_deg1 = Pipeline([
    ('scaler', StandardScaler()),
    ('logistic', LogisticRegression(penalty=None, solver='lbfgs'))
])
model_deg1.fit(X_train, y_train)

# Model 2: Degree 2 Polynomial Model (Curved Boundary)
model_deg2 = Pipeline([
    ('poly', PolynomialFeatures(degree=2)),
    ('scaler', StandardScaler()),
    ('logistic', LogisticRegression(penalty=None, solver='lbfgs'))
])
model_deg2.fit(X_train, y_train)

# Model 3: Degree 7 Polynomial Model (Overfitting Risk)
model_deg7 = Pipeline([
    ('poly', PolynomialFeatures(degree=7)),
    ('scaler', StandardScaler()),
    ('logistic', LogisticRegression(penalty=None, solver='lbfgs'))
])
model_deg7.fit(X_train, y_train)

# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot Degree 1
plot_decision_boundary(model_deg1, X_train, y_train, 'Degree 1: Linear Boundary\n(Underfitting)', axes[0])
# Plot Degree 2
plot_decision_boundary(model_deg2, X_train, y_train, 'Degree 2: Quadratic Boundary\n(Optimal Fit)', axes[1])
# Plot Degree 7
plot_decision_boundary(model_deg7, X_train, y_train, 'Degree 7: Complex Boundary\n(Overfitting Risk)', axes[2])

plt.tight_layout()
plt.show()

# Print accuracies
print(f"Degree 1 Test Accuracy: {accuracy_score(y_test, model_deg1.predict(X_test)):.4f}")
print(f"Degree 2 Test Accuracy: {accuracy_score(y_test, model_deg2.predict(X_test)):.4f}")
print(f"Degree 7 Test Accuracy: {accuracy_score(y_test, model_deg7.predict(X_test)):.4f}")

## 7. Demo C: Hyperparameter Tuning via GridSearchCV

Let's use `GridSearchCV` to automatically find the optimal values for regularization strength `C` and penalty type `penalty` for our model.

In [ ]:
# Create model pipeline
tuning_pipe = Pipeline([
    ('poly', PolynomialFeatures(degree=3)),
    ('scaler', StandardScaler()),
    ('logistic', LogisticRegression(solver='saga', max_iter=5000, random_state=42))
])

# Define grid of hyperparameters
# Note: solver='saga' is selected because it supports both L1 and L2 penalties
param_grid = {
    'logistic__C': [0.01, 0.1, 1.0, 10.0, 100.0],
    'logistic__penalty': ['l1', 'l2']
}

# Perform Grid Search with 5-fold cross-validation
grid_search = GridSearchCV(tuning_pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"Best Cross-Validation Accuracy: {grid_search.best_score_:.4f}")

# Evaluate best model on test set
best_model = grid_search.best_estimator_
test_acc = accuracy_score(y_test, best_model.predict(X_test))
print(f"Test Accuracy of Best Model: {test_acc:.4f}")

# Plot best model decision boundaries
plt.figure(figsize=(10, 6))
plot_decision_boundary(best_model, X_train, y_train, 'Best Tuned Model Decision Boundary')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.tight_layout()
plt.show()

## Summary Checklist for Advanced Logistic Regression

1. **Use Softmax for Multi-Class:** When classifying $K > 2$ target classes, set `multi_class='multinomial'` to utilize the Softmax activation function and Categorical Cross Entropy loss.
2. **Apply Polynomial Features for Curved Data:** Expand features using `PolynomialFeatures` to generate polynomial transformations, enabling curved boundaries.
3. **Control Overfitting:** Balance feature expansion degree and tune `C` (smaller values increase L1/L2 regularization) to prevent overfitting.
4. **Choose Solver Carefully:** Verify your chosen solver supports the intended penalty type (e.g. use `'saga'` for combined ElasticNet and L1 optimization).
5. **Scale Features:** Always standardize features using `StandardScaler` when regularizing or using Gradient-Descent based solvers to ensure correct convergence.